# Vehicle Title Image Scraper

Scrapes vehicle title images from multiple online sources and organizes them by US state.

**Sources:** eBay listings, DMV examples, auction sites (Copart/IAAI), title guides, and general web image search.

**Output:** Images organized into per-state folders with a `metadata.csv` tracking provenance.

## 1. Setup

In [ ]:
!pip install -q requests beautifulsoup4 ddgs Pillow tqdm

import os
import sys
import subprocess

# Detect environment
IN_COLAB = "google.colab" in str(get_ipython())
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/vehicle_titles"
    REPO_DIR = "/content/repo"
elif IN_KAGGLE:
    OUTPUT_DIR = "/kaggle/working/vehicle_titles"
    REPO_DIR = "/kaggle/working/repo"
else:
    OUTPUT_DIR = "./vehicle_titles"
    REPO_DIR = "."

# Clone repo and import scraper (skip if running locally in the repo)
REPO_URL = "https://github.com/nick-leland/Vehicle-Title-Scrape.git"

if REPO_DIR != ".":
    if os.path.exists(os.path.join(REPO_DIR, "scraper.py")):
        # Already cloned — pull latest
        print("Repo already cloned, pulling latest...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)
    else:
        # Fresh clone
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(["rm", "-rf", REPO_DIR], check=False)
        result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Clone failed: {result.stderr}")
            raise RuntimeError("Could not clone repo")
        print("Clone successful.")
    sys.path.insert(0, REPO_DIR)

from scraper import TitleImageScraper, US_STATES

print(f"\nEnvironment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"States available: {len(US_STATES)}")

## 2. Configuration

Edit these values before running. Set `STATES` to a short list for testing, or `None` for all 50.

In [ ]:
# --- EDIT THESE ---

# Which states to scrape. None = all 50. Use a short list for testing.
STATES = ["California", "Texas", "Florida"]

# Source types to include. None = all. Options: "general", "ebay", "dmv", "auction", "guide"
SOURCE_TYPES = None

# Seconds between HTTP requests (lower = faster but more likely to get rate-limited)
RATE_LIMIT = 3.0

# Max text search results per query (each result page is then scraped for images)
MAX_RESULTS_PER_QUERY = 10

## 3. Preview Search Queries

Dry-run to see what queries will be generated before starting the scrape.

In [ ]:
scraper = TitleImageScraper(
    output_dir=OUTPUT_DIR,
    rate_limit=RATE_LIMIT,
    max_results_per_query=MAX_RESULTS_PER_QUERY,
)

queries = scraper.build_search_queries(states=STATES, source_types=SOURCE_TYPES)

print(f"Total queries: {len(queries)}")
print(f"States: {sorted(set(q['state'] for q in queries))}")
print(f"Source types: {sorted(set(q['source_type'] for q in queries))}")
print()

# Show a sample
print("--- Sample queries ---")
for q in queries[:12]:
    print(f"  [{q['source_type']:>8}] {q['query']}")

## 4. Run the Scraper

This is the main scraping cell. It will:
1. Resume from any previous run (skips already-downloaded images via content hash)
2. Search via DuckDuckGo text search for pages containing title images
3. Scrape those pages for images, prioritizing ones with document-related keywords in alt/title/src
4. Download, validate (min 300px, document-like color profile, aspect ratio check), deduplicate, and save
5. Periodically save `metadata.csv`

**Expect this to take a while.** ~12 queries per state, 3s+ between requests, plus page scraping and download time.

In [ ]:
results = scraper.run(states=STATES, source_types=SOURCE_TYPES)
print("\n--- Results ---")
for k, v in results.items():
    print(f"  {k}: {v}")

## 5. Results Summary

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(OUTPUT_DIR, "metadata.csv"))
print(f"Total images: {len(df)}")
print()

print("--- Per state ---")
print(df["state"].value_counts().to_string())
print()

print("--- Per source type ---")
print(df["source_type"].value_counts().to_string())
print()

print("--- Image size stats ---")
print(df["file_size_bytes"].describe().to_string())
print()

print("--- First 10 records ---")
df.head(10)

## 6. Preview Downloaded Images

Display a sample grid of images per state to quickly assess quality.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from pathlib import Path

PREVIEW_PER_STATE = 6  # images to show per state

states_in_data = df["state"].unique()

for state in sorted(states_in_data):
    state_dir = Path(OUTPUT_DIR) / state.replace(" ", "_")
    if not state_dir.exists():
        continue

    image_files = sorted(state_dir.glob("*"))[:PREVIEW_PER_STATE]
    if not image_files:
        continue

    n = len(image_files)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle(f"{state} ({len(list(state_dir.glob('*')))} total)", fontsize=14)

    for ax, img_path in zip(axes, image_files):
        try:
            img = PILImage.open(img_path)
            ax.imshow(img)
            ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=9)
        except Exception:
            ax.text(0.5, 0.5, "Error", ha="center", va="center")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 7. State Distribution Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Per-state counts
state_counts = df["state"].value_counts()
state_counts.plot.barh(ax=ax1)
ax1.set_xlabel("Images")
ax1.set_title("Images per State")
ax1.invert_yaxis()

# Per-source counts
source_counts = df["source_type"].value_counts()
source_counts.plot.bar(ax=ax2, rot=30)
ax2.set_ylabel("Images")
ax2.set_title("Images per Source Type")

plt.tight_layout()
plt.show()

## 8. Manual Curation

Use the cells below to flag images for removal. After reviewing the previews above, add filenames or hashes to the removal list and re-run to clean up.

In [ ]:
# Add filenames to remove (from the preview or metadata inspection)
REMOVE_FILES = [
    # "a1b2c3d4e5f6abcd.jpg",
]

if REMOVE_FILES:
    removed = 0
    for fname in REMOVE_FILES:
        # Find and delete the file
        matches = list(Path(OUTPUT_DIR).rglob(fname))
        for m in matches:
            m.unlink()
            removed += 1

        # Remove from metadata
        df = df[df["filename"] != fname]

    # Save updated metadata
    df.to_csv(os.path.join(OUTPUT_DIR, "metadata.csv"), index=False)
    print(f"Removed {removed} files. Metadata updated ({len(df)} remaining).")
else:
    print("Nothing to remove. Add filenames to REMOVE_FILES above.")

## 9. Resume / Extend

Re-run the scraper for additional states or source types. Already-downloaded images are automatically skipped via content hash dedup.

In [ ]:
# Add more states here and re-run this cell
ADDITIONAL_STATES = [
    # "New York", "Ohio", "Michigan",
]

if ADDITIONAL_STATES:
    extra_results = scraper.run(states=ADDITIONAL_STATES, source_types=SOURCE_TYPES)
    print("\n--- Additional run results ---")
    for k, v in extra_results.items():
        print(f"  {k}: {v}")
else:
    print("Set ADDITIONAL_STATES above to scrape more states.")